# Vector-Quantized Variational Autoencoders (VQ-VAE)

## Introduction

**Vector-Quantized Variational Autoencoders (VQ-VAEs)** represent a fascinating evolution in generative modeling that replaces continuous latent spaces with **discrete latent representations**. Instead of encoding images as continuous vectors like VAEs, VQ-VAEs learn a **codebook** of discrete vectors and represent images as combinations of these learned codes.

This discrete approach offers several powerful advantages:

1. **No posterior collapse**: A common VAE problem where the model ignores the latent space
2. **Sharper reconstructions**: Discrete codes can capture more precise patterns
3. **Hierarchical generation**: Perfect for autoregressive models like PixelCNN
4. **Interpretable representations**: Each code can represent a meaningful pattern

In this notebook, we'll build deep intuitions about VQ-VAEs by:
- Understanding why discrete latent spaces are powerful
- Implementing vector quantization from scratch
- Training a VQ-VAE with PyTorch Lightning
- Analyzing learned codebooks and their usage patterns
- Exploring applications and comparisons with VAEs

**Prerequisites**: This notebook assumes familiarity with VAEs. If you haven't seen the VAE notebook yet, start there first!

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pytorch_lightning as pl

from aiml_notebooks import (
    get_device, set_seed, count_parameters,
)

%load_ext autoreload
%autoreload 2

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(42)
device = get_device()

## Data Preparation

We'll use **MNIST** to focus on understanding VQ-VAE mechanics without getting distracted by complex image processing.

In [ ]:
# Transform - just convert to tensor (images already in [0, 1])
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Image value range: [0, 1] (unnormalized)")

Visualize some samples to understand our data.

In [ ]:
# Get a batch of training data
sample_batch, sample_labels = next(iter(train_loader))

# Plot first 16 images
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"Label: {sample_labels[i].item()}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## Why Discrete Latent Spaces?

### The Challenge with Continuous Latent Spaces

VAEs use continuous latent spaces: $z \in \mathbb{R}^d$. While powerful, they have limitations:

1. **Posterior collapse**: The model may ignore the latent space and rely only on the decoder
2. **Blurry reconstructions**: Continuous interpolation can average out sharp features
3. **Limited expressiveness**: Real-world data often has discrete structure (e.g., object parts)

### The VQ-VAE Solution: Discrete Codes

Instead of continuous $z$, VQ-VAE uses a **codebook** $\mathcal{E} = \{e_1, e_2, ..., e_K\}$ where each $e_k \in \mathbb{R}^d$.

**Key idea**: The encoder outputs a continuous vector $z_e$, but we **quantize** it to the nearest codebook vector $e_k$:

$$z_q = e_k \quad \text{where} \quad k = \arg\min_j \|z_e - e_j\|_2$$

The decoder then reconstructs from the **discrete** code $z_q$.

**Benefits**:
- **No posterior collapse**: Discrete codes force the model to use the latent space
- **Sharp reconstructions**: Discrete representations preserve fine details
- **Compositionality**: Combine discrete codes hierarchically
- **Perfect for autoregressive models**: Can model $p(z)$ with PixelCNN/Transformers

## Understanding Vector Quantization

Let's build intuition with a simple 2D example before implementing the full model.

### Concept

Imagine we have:
- **Continuous points** in 2D space (encoder outputs)
- **Codebook of 8 vectors** (learned discrete representations)
- **Quantization**: Map each continuous point to its nearest codebook vector

In [ ]:
# Create some random continuous points
np.random.seed(42)
continuous_points = np.random.randn(100, 2)

# Create a small codebook (8 vectors in 2D)
codebook = np.array([
    [-2, -2], [-2, 0], [-2, 2],
    [0, -2], [0, 2],
    [2, -2], [2, 0], [2, 2]
])

# Quantize: find nearest codebook vector for each point
def quantize(points, codebook):
    """Find nearest codebook vector for each point."""
    # Compute distances: (n_points, n_codes)
    distances = np.sum((points[:, None, :] - codebook[None, :, :]) ** 2, axis=2)
    # Find nearest
    indices = np.argmin(distances, axis=1)
    # Get quantized vectors
    quantized = codebook[indices]
    return quantized, indices

quantized_points, code_indices = quantize(continuous_points, codebook)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before quantization
axes[0].scatter(continuous_points[:, 0], continuous_points[:, 1], alpha=0.6, s=50, c='blue')
axes[0].scatter(codebook[:, 0], codebook[:, 1], s=200, c='red', marker='X', edgecolors='black', linewidths=2)
axes[0].set_title('Before Quantization\n(continuous points + codebook)', fontsize=12)
axes[0].set_xlabel('Dimension 1')
axes[0].set_ylabel('Dimension 2')
axes[0].grid(True, alpha=0.3)
axes[0].legend(['Continuous points', 'Codebook vectors'], loc='upper right')

# After quantization
axes[1].scatter(quantized_points[:, 0], quantized_points[:, 1], alpha=0.6, s=50, c=code_indices, cmap='tab10')
axes[1].scatter(codebook[:, 0], codebook[:, 1], s=200, c='red', marker='X', edgecolors='black', linewidths=2)
axes[1].set_title('After Quantization\n(discrete codes)', fontsize=12)
axes[1].set_xlabel('Dimension 1')
axes[1].set_ylabel('Dimension 2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Original points: {len(continuous_points)} unique positions")
print(f"Quantized points: {len(np.unique(code_indices))} unique codes used")
print(f"Compression: {len(continuous_points)} → {len(np.unique(code_indices))} codes")

**Key insight**: Vector quantization maps a continuous space to a discrete set of learned representatives. Each continuous point "snaps" to its nearest codebook vector.

## The VQ-VAE Architecture

Now let's understand the complete VQ-VAE pipeline:

1. **Encoder**: Maps input $x$ to continuous representation $z_e$
2. **Vector Quantization**: Quantizes $z_e$ to nearest codebook vector $z_q$
3. **Decoder**: Reconstructs from $z_q$ back to $\hat{x}$

### The VQ-VAE Loss (with EMA)

With EMA codebook updates, VQ-VAE optimizes two objectives:

$$\mathcal{L} = \underbrace{\|x - \hat{x}\|_2^2}_\text{Reconstruction} + \beta \underbrace{\|z_e - \text{sg}[e]\|_2^2}_\text{Commitment Loss}$$

Where $\text{sg}[\cdot]$ is the stop-gradient operator (no backprop through it).

**What each term does:**

1. **Reconstruction loss**: Standard autoencoder objective - reconstruct the input
2. **Commitment loss**: Encourages encoder outputs to stay close to chosen codebook vectors

**Note**: There's NO codebook loss term when using EMA! The codebook is updated via:
- **Exponential Moving Average** of encoder outputs assigned to each code
- Not through backpropagation

### The Straight-Through Estimator

**Problem**: Quantization (choosing nearest neighbor) is **not differentiable**!

**Solution**: Copy gradients from decoder straight through to encoder:

$$\frac{\partial \mathcal{L}}{\partial z_e} = \frac{\partial \mathcal{L}}{\partial z_q}$$

In code: `z_q = z_e + (z_q - z_e).detach()`

This clever trick allows gradients to flow while maintaining discrete quantization!

### Summary: Two Key Techniques

1. **EMA updates**: For codebook (no gradients, just moving average)
2. **Straight-through estimator**: For encoder gradients (copy through quantization)

## Building the VQ-VAE: Encoder

The encoder is similar to a regular autoencoder - it maps inputs to latent representations. The key difference: its outputs will be **quantized** to discrete codes.

In [ ]:
class VQVAEEncoder(nn.Module):
    """Encoder that maps input to continuous latent representation (before quantization)."""
    
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, x):
        # Flatten image: (batch, 1, 28, 28) -> (batch, 784)
        x = x.view(x.size(0), -1)
        h = F.relu(self.fc1(x))
        z_e = self.fc2(h)  # Continuous latent representation
        return z_e

Test the encoder with sample data.

In [ ]:
# Create encoder
encoder = VQVAEEncoder(latent_dim=20).to(device)
print(f"Encoder parameters: {count_parameters(encoder):,}")

# Test with sample batch
test_input = sample_batch[:4].to(device)
z_e = encoder(test_input)

print(f"\nInput shape: {test_input.shape}")
print(f"Encoder output (z_e) shape: {z_e.shape}")
print(f"\nSample z_e values: {z_e[0, :5].detach().cpu().numpy()}")

## Building the Vector Quantizer with EMA Updates

This is the heart of VQ-VAE! The vector quantizer:
1. Maintains a learnable codebook of $K$ vectors
2. Maps each encoder output $z_e$ to its nearest codebook vector
3. **Updates codebook using EMA** (Exponential Moving Average)
4. Computes commitment loss
5. Uses straight-through estimator for gradients

### EMA Codebook Updates (Original VQ-VAE)

Instead of using gradients to update the codebook, we use **Exponential Moving Average**:

**Update cluster size:**
$$N_i^{(t)} = \gamma \cdot N_i^{(t-1)} + (1-\gamma) \cdot n_i^{(t)}$$

where $n_i^{(t)}$ = number of vectors assigned to code $i$ in current batch

**Update embedding sum:**
$$m_i^{(t)} = \gamma \cdot m_i^{(t-1)} + (1-\gamma) \cdot \sum_{j: \text{code}(z_j)=i} z_j^{(t)}$$

**Normalize to get codebook vector:**
$$e_i^{(t)} = \frac{m_i^{(t)}}{N_i^{(t)}}$$

**Benefits of EMA:**
- **No learning rate for codebook**: More stable training
- **Better utilization**: Codes naturally track their assigned regions
- **No codebook loss term**: Only commitment loss needed

With EMA, the loss becomes simpler:
$$\mathcal{L} = \underbrace{\|x - \hat{x}\|_2^2}_\text{Reconstruction} + \beta \underbrace{\|z_e - \text{sg}[e]\|_2^2}_\text{Commitment Loss}$$

No codebook loss needed - EMA handles it!

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantization layer with EMA codebook updates (original VQ-VAE)."""
    
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25, decay=0.99, epsilon=1e-5):
        super().__init__()
        
        self.num_embeddings = num_embeddings  # Size of codebook (K)
        self.embedding_dim = embedding_dim    # Dimension of each code vector
        self.commitment_cost = commitment_cost  # Beta in commitment loss
        
        # EMA parameters
        self.decay = decay
        self.epsilon = epsilon
        
        # Initialize codebook embeddings
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)
        
        # EMA cluster size and embeddings (not part of model parameters - use register_buffer)
        self.register_buffer('ema_cluster_size', torch.zeros(num_embeddings))
        self.register_buffer('ema_weight', self.embedding.weight.data.clone())
    
    def forward(self, z_e):
        """
        Args:
            z_e: Encoder output (batch_size, embedding_dim)
        
        Returns:
            z_q: Quantized vectors (same shape as z_e)
            vq_loss: Vector quantization loss (commitment only - no codebook loss with EMA)
            perplexity: Measure of codebook usage diversity
            encoding_indices: Which codebook vectors were chosen
        """
        # Flatten input if needed
        input_shape = z_e.shape
        z_e_flat = z_e.view(-1, self.embedding_dim)  # (batch_size, embedding_dim)
        
        # Calculate distances to all codebook vectors
        # ||z_e - e||^2 = ||z_e||^2 + ||e||^2 - 2 * z_e · e
        distances = (
            torch.sum(z_e_flat**2, dim=1, keepdim=True)  # ||z_e||^2
            + torch.sum(self.embedding.weight**2, dim=1)  # ||e||^2
            - 2 * torch.matmul(z_e_flat, self.embedding.weight.t())  # -2 * z_e · e
        )  # Shape: (batch_size, num_embeddings)
        
        # Find nearest codebook vectors
        encoding_indices = torch.argmin(distances, dim=1)  # (batch_size,)
        
        # Convert to one-hot for getting quantized vectors
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()
        
        # Quantize: get the actual codebook vectors
        z_q_flat = torch.matmul(encodings, self.embedding.weight)  # (batch_size, embedding_dim)
        z_q = z_q_flat.view(input_shape)
        
        # Update codebook using EMA (only during training)
        if self.training:
            # Update cluster size with EMA
            # N_i = decay * N_i + (1 - decay) * n_i
            # where n_i = number of vectors assigned to code i in this batch
            encodings_sum = encodings.sum(0)  # (num_embeddings,)
            self.ema_cluster_size.data.mul_(self.decay).add_(
                encodings_sum, alpha=1 - self.decay
            )
            
            # Laplace smoothing to prevent cluster size from going to zero
            n = self.ema_cluster_size.sum()
            self.ema_cluster_size.data.add_(self.epsilon).div_(n + self.num_embeddings * self.epsilon).mul_(n)
            
            # Update embeddings with EMA
            # m_i = decay * m_i + (1 - decay) * sum(z_e where code = i)
            dw = torch.matmul(encodings.t(), z_e_flat)  # (num_embeddings, embedding_dim)
            self.ema_weight.data.mul_(self.decay).add_(dw, alpha=1 - self.decay)
            
            # Normalize: e_i = m_i / N_i
            self.embedding.weight.data.copy_(
                self.ema_weight / self.ema_cluster_size.unsqueeze(1)
            )
        
        # Compute loss
        # With EMA, we only have commitment loss (no codebook loss - it's updated via EMA)
        # Commitment loss: ||z_e - sg[e]||^2 (keeps encoder committed to chosen codes)
        commitment_loss = F.mse_loss(z_e, z_q.detach())
        vq_loss = self.commitment_cost * commitment_loss
        
        # Straight-through estimator: copy gradients from decoder to encoder
        z_q = z_e + (z_q - z_e).detach()
        
        # Perplexity: measure of how many codebook vectors are actively used
        # High perplexity = diverse usage, Low perplexity = codebook collapse
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        return z_q, vq_loss, perplexity, encoding_indices

Test the vector quantizer to understand its behavior.

In [ ]:
# Create vector quantizer
vq = VectorQuantizer(num_embeddings=512, embedding_dim=20).to(device)
print(f"Codebook size: {vq.num_embeddings} vectors")
print(f"Codebook dimension: {vq.embedding_dim}")
print(f"Total codebook parameters: {count_parameters(vq):,}")

# Test with encoder output
z_q, vq_loss, perplexity, indices = vq(z_e)

print(f"\nEncoder output (z_e) shape: {z_e.shape}")
print(f"Quantized output (z_q) shape: {z_q.shape}")
print(f"VQ loss: {vq_loss.item():.4f}")
print(f"Perplexity: {perplexity.item():.2f} / {vq.num_embeddings}")
print(f"Codebook indices used: {indices.cpu().numpy()}")
print(f"\nBefore quantization (z_e): {z_e[0, :5].detach().cpu().numpy()}")
print(f"After quantization (z_q): {z_q[0, :5].detach().cpu().numpy()}")

Notice how the quantized values $z_q$ differ from the encoder outputs $z_e$ - they've been "snapped" to the nearest codebook vectors!

## Building the VQ-VAE: Decoder

The decoder maps quantized codes back to reconstructed images. It's structurally similar to a regular autoencoder decoder.

In [ ]:
class VQVAEDecoder(nn.Module):
    """Decoder that maps quantized latent code to reconstructed image."""
    
    def __init__(self, latent_dim=20, hidden_dim=400, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, z_q):
        h = F.relu(self.fc1(z_q))
        x_recon = torch.sigmoid(self.fc2(h))  # Sigmoid for [0, 1] pixel values
        # Reshape to image: (batch, 784) -> (batch, 1, 28, 28)
        x_recon = x_recon.view(x_recon.size(0), 1, 28, 28)
        return x_recon

Test the decoder with quantized codes.

In [ ]:
# Create decoder
decoder = VQVAEDecoder(latent_dim=20).to(device)
print(f"Decoder parameters: {count_parameters(decoder):,}")

# Test with quantized codes
test_recon = decoder(z_q)

print(f"\nQuantized code (z_q) shape: {z_q.shape}")
print(f"Reconstruction shape: {test_recon.shape}")

## Complete VQ-VAE Model with PyTorch Lightning

Now we combine encoder, vector quantizer, and decoder into a complete VQ-VAE using **PyTorch Lightning** for clean, organized training.

The complete forward pass:
1. Encode: $x \rightarrow z_e$
2. Quantize: $z_e \rightarrow z_q$ (nearest codebook vector)
3. Decode: $z_q \rightarrow \hat{x}$

In [ ]:
class VQVAE(pl.LightningModule):
    """Complete VQ-VAE with PyTorch Lightning."""
    
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20, 
                 num_embeddings=512, commitment_cost=0.25, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()
        
        # Components
        self.encoder = VQVAEEncoder(input_dim, hidden_dim, latent_dim)
        self.vq_layer = VectorQuantizer(num_embeddings, latent_dim, commitment_cost)
        self.decoder = VQVAEDecoder(latent_dim, hidden_dim, input_dim)
        
        self.latent_dim = latent_dim
        self.num_embeddings = num_embeddings
    
    def encode(self, x):
        """Encode input to continuous latent."""
        return self.encoder(x)
    
    def quantize(self, z_e):
        """Quantize continuous latent to discrete codes."""
        return self.vq_layer(z_e)
    
    def decode(self, z_q):
        """Decode quantized latent to reconstruction."""
        return self.decoder(z_q)
    
    def forward(self, x):
        """Full forward pass: encode -> quantize -> decode."""
        z_e = self.encode(x)
        z_q, vq_loss, perplexity, indices = self.quantize(z_e)
        x_recon = self.decode(z_q)
        return x_recon, vq_loss, perplexity, indices
    
    def training_step(self, batch, batch_idx):
        """Training step - called for each batch during training."""
        x, _ = batch
        
        # Forward pass
        x_recon, vq_loss, perplexity, _ = self(x)
        
        # Reconstruction loss (MSE)
        recon_loss = F.mse_loss(x_recon, x)
        
        # Total loss
        loss = recon_loss + vq_loss
        
        # Log metrics
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_recon', recon_loss, prog_bar=True)
        self.log('train_vq', vq_loss, prog_bar=True)
        self.log('train_perplexity', perplexity, prog_bar=False)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        """Validation step - called for each batch during validation."""
        x, _ = batch
        
        # Forward pass
        x_recon, vq_loss, perplexity, _ = self(x)
        
        # Reconstruction loss
        recon_loss = F.mse_loss(x_recon, x)
        
        # Total loss
        loss = recon_loss + vq_loss
        
        # Log metrics
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_recon', recon_loss, prog_bar=True)
        self.log('val_vq', vq_loss, prog_bar=True)
        self.log('val_perplexity', perplexity, prog_bar=True)
        
        return loss
    
    def configure_optimizers(self):
        """Configure optimizer - Lightning handles the training loop."""
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer
    
    def sample_from_codebook(self, num_samples, device):
        """Sample random codes from codebook and decode them."""
        # Randomly select codebook indices
        random_indices = torch.randint(0, self.num_embeddings, (num_samples,), device=device)
        # Get corresponding codebook vectors
        z_q = self.vq_layer.embedding(random_indices)
        # Decode
        samples = self.decode(z_q)
        return samples

Create the VQ-VAE model and verify its structure.

In [ ]:
# Create VQ-VAE
latent_dim = 20
num_embeddings = 512

vqvae = VQVAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=latent_dim,
    num_embeddings=num_embeddings,
    commitment_cost=0.25,
    learning_rate=1e-3
)

print(f"Total parameters: {count_parameters(vqvae):,}")
print(f"Encoder parameters: {count_parameters(vqvae.encoder):,}")
print(f"Codebook parameters: {count_parameters(vqvae.vq_layer):,}")
print(f"Decoder parameters: {count_parameters(vqvae.decoder):,}")

# Test forward pass (model starts on CPU)
test_input = sample_batch[:4]
test_recon, test_vq_loss, test_perplexity, test_indices = vqvae(test_input)

print(f"\nInput shape: {test_input.shape}")
print(f"Reconstruction shape: {test_recon.shape}")
print(f"VQ loss: {test_vq_loss.item():.4f}")
print(f"Perplexity: {test_perplexity.item():.2f}")

## Training the VQ-VAE

Now we'll train the VQ-VAE on MNIST using PyTorch Lightning's `Trainer`.

**What to watch during training:**
- **Reconstruction loss**: Should decrease (model learning to reconstruct)
- **VQ loss**: Should stabilize (codebook and encoder converging)
- **Perplexity**: Should remain reasonably high (diverse codebook usage)

Low perplexity indicates **codebook collapse** - only a few codes being used. Healthy VQ-VAE maintains high perplexity throughout training.

In [ ]:
# Training setup
num_epochs = 10

# Recreate model to start fresh
vqvae = VQVAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=latent_dim,
    num_embeddings=num_embeddings,
    commitment_cost=0.25,
    learning_rate=1e-3
)

# Create Lightning Trainer
trainer = pl.Trainer(
    max_epochs=num_epochs,
    accelerator='auto',  # Automatically uses MPS/CUDA/CPU
    devices=1,
    logger=False,  # Disable default logger for cleaner output
    enable_checkpointing=False,
    enable_progress_bar=True,
)

print(f"Training VQ-VAE for {num_epochs} epochs with PyTorch Lightning...\n")

# Train the model - Lightning handles everything!
trainer.fit(vqvae, train_loader, test_loader)

print("\nTraining complete!")

## Improving VQ-VAE Performance

You may have noticed the reconstructions aren't great and perplexity is low (~7.6% codebook usage). This is **codebook collapse** - a common VQ-VAE problem!

### Why does this happen?

1. **Simple MLP architecture**: Flattening images loses spatial structure
2. **Codebook too large**: 512 codes is too many for this simple architecture
3. **Not enough training**: 10 epochs may be insufficient for EMA to converge

### Fixes

We're already using **EMA (Exponential Moving Average)** updates from the original VQ-VAE paper, which is better than gradient-based updates. But we can improve further:

1. **Smaller codebook**: 128 codes instead of 512
2. **More training**: 20 epochs for EMA to converge
3. **Standard commitment**: 0.25 is fine with EMA

The key insight: **EMA works better with smaller codebooks** because each code gets more training examples.

Let's train an improved version!

In [ ]:
# Improved VQ-VAE with better hyperparameters
print("Training improved VQ-VAE with EMA...\n")

vqvae_improved = VQVAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=20,
    num_embeddings=128,  # Smaller codebook (was 512)
    commitment_cost=0.25,   # Standard commitment with EMA
    learning_rate=1e-3
)

# Train for longer
trainer_improved = pl.Trainer(
    max_epochs=20,
    accelerator='auto',
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)

trainer_improved.fit(vqvae_improved, train_loader, test_loader)

print("\nImproved training complete!")

Let's compare the improved version with the original.

In [ ]:
# Compare reconstructions
test_batch_comp, _ = next(iter(test_loader))
test_batch_comp = test_batch_comp.to(vqvae_improved.device)

vqvae.eval()
vqvae_improved.eval()

with torch.no_grad():
    recon_original, _, perp_orig, _ = vqvae(test_batch_comp)
    recon_improved, _, perp_impr, _ = vqvae_improved(test_batch_comp)

# Visualize comparison
n_samples = 6
fig, axes = plt.subplots(3, n_samples, figsize=(14, 6))

for i in range(n_samples):
    # Original image
    axes[0, i].imshow(test_batch_comp[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=12)
    
    # Original VQ-VAE
    axes[1, i].imshow(recon_original[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel(f'Original VQ-VAE\n({perp_orig.item():.1f}/{vqvae.num_embeddings} codes)', fontsize=10)
    
    # Improved VQ-VAE
    axes[2, i].imshow(recon_improved[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel(f'Improved VQ-VAE\n({perp_impr.item():.1f}/{vqvae_improved.num_embeddings} codes)', fontsize=10)

plt.suptitle('Comparison: Original vs Improved VQ-VAE', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nOriginal VQ-VAE:")
print(f"  Codebook size: {vqvae.num_embeddings}")
print(f"  Perplexity: {perp_orig.item():.2f}")
print(f"  Codebook usage: {perp_orig.item()/vqvae.num_embeddings*100:.1f}%")

print(f"\nImproved VQ-VAE:")
print(f"  Codebook size: {vqvae_improved.num_embeddings}")
print(f"  Perplexity: {perp_impr.item():.2f}")
print(f"  Codebook usage: {perp_impr.item()/vqvae_improved.num_embeddings*100:.1f}%")
print(f"\nNotice the improved reconstructions and better codebook utilization!")

## Evaluating Reconstructions

Let's see how well the VQ-VAE reconstructs test images. Since the latent space is **discrete**, we expect sharp reconstructions that may differ slightly from the originals.

In [ ]:
# Get test batch
test_batch, test_labels = next(iter(test_loader))
test_batch = test_batch.to(vqvae.device)

# Reconstruct
vqvae.eval()
with torch.no_grad():
    recon_batch, vq_loss, perplexity, indices = vqvae(test_batch)

# Visualize
n_samples = 8
fig, axes = plt.subplots(2, n_samples, figsize=(16, 4))

for i in range(n_samples):
    # Original
    axes[0, i].imshow(test_batch[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=12)
    
    # Reconstruction
    axes[1, i].imshow(recon_batch[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Code: {indices[i].item()}', fontsize=9)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Reconstructed', fontsize=12)

plt.suptitle('VQ-VAE Reconstructions (with codebook indices)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Perplexity: {perplexity.item():.2f} / {num_embeddings}")
print(f"Codebook usage: {perplexity.item() / num_embeddings * 100:.1f}%")

Notice the codebook indices shown above each reconstruction - these are the **discrete codes** that represent each image!

## Analyzing Codebook Usage

One of the most interesting aspects of VQ-VAE is understanding which codebook vectors are actually used and how often.

**Healthy codebook usage:**
- Many different codes are used
- High perplexity (approaching the codebook size)
- Relatively balanced distribution

**Codebook collapse:**
- Only a few codes dominate
- Low perplexity
- Most codebook vectors unused

In [ ]:
def analyze_codebook_usage(model, dataloader, device):
    """Analyze which codebook vectors are used and how often."""
    model.eval()
    
    # Count usage of each codebook vector
    codebook_counts = torch.zeros(model.num_embeddings)
    
    with torch.no_grad():
        for data, _ in tqdm(dataloader, desc='Analyzing codebook'):
            data = data.to(device)
            _, _, _, indices = model(data)
            
            # Count occurrences
            for idx in indices:
                codebook_counts[idx.item()] += 1
    
    return codebook_counts.numpy()

# Analyze codebook usage on test set
codebook_usage = analyze_codebook_usage(vqvae, test_loader, vqvae.device)

# Visualize
plt.figure(figsize=(14, 5))
plt.bar(range(len(codebook_usage)), codebook_usage, width=1.0, alpha=0.7)
plt.xlabel('Codebook Index')
plt.ylabel('Usage Count')
plt.title('VQ-VAE Codebook Vector Usage on Test Set')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Statistics
used_vectors = np.sum(codebook_usage > 0)
print(f"Total codebook size: {num_embeddings}")
print(f"Vectors used: {used_vectors} ({used_vectors/num_embeddings*100:.1f}%)")
print(f"Unused vectors: {num_embeddings - used_vectors}")
print(f"Most used vector: {np.max(codebook_usage):.0f} times")
print(f"Average usage (used vectors): {np.mean(codebook_usage[codebook_usage > 0]):.1f} times")

### Visualizing Most Common Codebook Vectors

Let's decode the most frequently used codebook vectors to see what patterns they represent!

In [ ]:
# Get top 16 most-used codebook vectors
top_indices = np.argsort(codebook_usage)[-16:][::-1].copy()

# Decode them
vqvae.eval()
with torch.no_grad():
    # Get codebook vectors
    top_codes = torch.tensor(top_indices, device=vqvae.device)
    z_q = vqvae.vq_layer.embedding(top_codes)
    # Decode
    decoded_images = vqvae.decode(z_q)

# Visualize
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(decoded_images[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Code {top_indices[i]}\n({codebook_usage[top_indices[i]]:.0f} uses)', fontsize=9)
    ax.axis('off')

plt.suptitle('Most Frequently Used Codebook Vectors (decoded to images)', fontsize=14)
plt.tight_layout()
plt.show()

print("These are the most common discrete representations learned by VQ-VAE!")
print("Each represents a typical pattern that appears frequently in the dataset.")

**Key insight**: The most-used codebook vectors often represent **common digit patterns** or **digit prototypes**. This shows how VQ-VAE learns a discrete vocabulary of visual patterns!

## Exploring the Discrete Latent Space

Unlike VAEs with continuous latent spaces, VQ-VAE has a **discrete** latent space. Let's explore what this means by looking at how different images map to codebook vectors.

In [ ]:
def get_code_distribution_by_digit(model, dataset, device, max_samples_per_digit=500):
    """Get distribution of codebook indices for each digit class."""
    model.eval()
    
    # Store which codes are used for each digit
    digit_to_codes = {i: [] for i in range(10)}
    counts_per_digit = {i: 0 for i in range(10)}
    
    with torch.no_grad():
        for data, label in tqdm(dataset, desc='Mapping digits to codes'):
            digit = label if isinstance(label, int) else label.item()
            
            # Limit samples per digit
            if counts_per_digit[digit] >= max_samples_per_digit:
                continue
            
            data = data.unsqueeze(0).to(device)
            _, _, _, indices = model(data)
            digit_to_codes[digit].append(indices.item())
            counts_per_digit[digit] += 1
            
            # Stop if we have enough samples for all digits
            if all(c >= max_samples_per_digit for c in counts_per_digit.values()):
                break
    
    return digit_to_codes

# Get code distribution for each digit
digit_to_codes = get_code_distribution_by_digit(vqvae, test_dataset, vqvae.device)

# Visualize distribution for each digit
fig, axes = plt.subplots(2, 5, figsize=(16, 6))

for digit in range(10):
    ax = axes[digit // 5, digit % 5]
    codes = digit_to_codes[digit]
    
    # Count code occurrences
    code_counts = np.bincount(codes, minlength=num_embeddings)
    
    # Plot only non-zero codes
    used_codes = np.where(code_counts > 0)[0]
    ax.bar(used_codes, code_counts[used_codes], width=1.0, alpha=0.7)
    ax.set_title(f'Digit {digit}', fontsize=12)
    ax.set_xlabel('Codebook Index', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.grid(alpha=0.3, axis='y')
    
    # Print stats
    unique_codes = len(used_codes)
    ax.text(0.95, 0.95, f'{unique_codes} codes',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Codebook Usage by Digit Class', fontsize=14)
plt.tight_layout()
plt.show()

# Summary statistics
print("\nCodes used per digit:")
for digit in range(10):
    unique_codes = len(set(digit_to_codes[digit]))
    print(f"  Digit {digit}: {unique_codes} unique codes")

**Key observations:**
- Each digit class uses a **subset** of the codebook
- Some codes are **shared** between similar digits (e.g., 3 and 8)
- Some codes are **specific** to particular digits
- This shows how VQ-VAE learns a **compositional** discrete vocabulary!

## Generating Samples

With VQ-VAE, we can generate samples in two ways:

1. **Random sampling from codebook**: Pick random codes and decode them
2. **Learned prior** (advanced): Train a separate model (like PixelCNN) to model $p(z)$

Let's try the first approach - randomly sampling codes from our learned codebook.

In [ ]:
# Generate samples by randomly selecting codebook vectors
num_samples = 16

vqvae.eval()
with torch.no_grad():
    samples = vqvae.sample_from_codebook(num_samples, vqvae.device)

# Visualize
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    ax.axis('off')

plt.suptitle('Random Samples from Codebook', fontsize=14)
plt.tight_layout()
plt.show()

print("Note: These are random codes, not from a learned prior.")
print("For better generation, train a prior model (like PixelCNN) on the codes!")

The random samples may not look perfect because we haven't learned a **prior distribution** $p(z)$ over the codes. In practice, VQ-VAE is often used as a first stage, with a second autoregressive model (like PixelCNN or Transformer) learning to generate sequences of codes.

## Latent Space Interpolation

Unlike VAEs with smooth continuous latent spaces, VQ-VAE has **discrete** codes. We can still "interpolate" by:
1. Encoding two images to get their codes
2. Creating a path through codebook space

Note: Since the space is discrete, interpolation will show **discrete transitions** rather than smooth morphing.

In [ ]:
# Select two different digits
idx1, idx2 = 0, 7
img1 = test_batch[idx1:idx1+1]
img2 = test_batch[idx2:idx2+1]
label1 = test_labels[idx1].item()
label2 = test_labels[idx2].item()

# Encode both to get their codes
vqvae.eval()
with torch.no_grad():
    z_e1 = vqvae.encode(img1)
    z_q1, _, _, idx_1 = vqvae.quantize(z_e1)
    
    z_e2 = vqvae.encode(img2)
    z_q2, _, _, idx_2 = vqvae.quantize(z_e2)
    
    # Continuous interpolation in embedding space (before quantization)
    n_steps = 10
    interpolations = []
    
    for alpha in np.linspace(0, 1, n_steps):
        # Interpolate in continuous space
        z_interp = (1 - alpha) * z_e1 + alpha * z_e2
        # Quantize
        z_q_interp, _, _, _ = vqvae.quantize(z_interp)
        # Decode
        img_interp = vqvae.decode(z_q_interp)
        interpolations.append(img_interp)
    
    interpolations = torch.cat(interpolations, dim=0).cpu()

# Visualize
fig, axes = plt.subplots(1, n_steps, figsize=(16, 2))
for i in range(n_steps):
    axes[i].imshow(interpolations[i].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i].axis('off')
    if i == 0:
        axes[i].set_title(f'Digit {label1}', fontsize=10)
    elif i == n_steps - 1:
        axes[i].set_title(f'Digit {label2}', fontsize=10)

plt.suptitle('VQ-VAE Latent Interpolation (discrete transitions)', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"Code for img1: {idx_1.item()}")
print(f"Code for img2: {idx_2.item()}")
print("\nNotice the discrete 'jumps' between codebook vectors rather than smooth transitions.")

## Comparing Codebook Sizes

The codebook size (number of discrete codes) is a crucial hyperparameter. Let's train VQ-VAEs with different codebook sizes to understand the tradeoff.

**Intuition:**
- **Small codebook** (e.g., 64 codes): Less expressive, may lose details
- **Large codebook** (e.g., 1024 codes): More expressive, but may underutilize codes

Let's compare!

In [ ]:
# Train VQ-VAEs with different codebook sizes
codebook_sizes = [64, 256, 512]
models = {}

for size in codebook_sizes:
    print(f"\nTraining VQ-VAE with codebook size {size}...")
    
    # Create model
    model = VQVAE(
        input_dim=784,
        hidden_dim=400,
        latent_dim=20,
        num_embeddings=size,
        commitment_cost=0.25,
        learning_rate=1e-3
    )
    
    # Train
    trainer = pl.Trainer(
        max_epochs=5,
        accelerator='auto',
        devices=1,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=True,
    )
    
    trainer.fit(model, train_loader, test_loader)
    models[size] = model

print("\nTraining complete for all codebook sizes!")

Compare reconstructions across different codebook sizes.

In [ ]:
# Get a test image
test_img = test_batch[0:1]

# Compare reconstructions
fig, axes = plt.subplots(1, len(codebook_sizes) + 1, figsize=(12, 3))

# Original
axes[0].imshow(test_img.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
axes[0].axis('off')

# Reconstructions
for i, size in enumerate(codebook_sizes):
    model = models[size]
    model.eval()
    with torch.no_grad():
        recon, _, perplexity, _ = model(test_img.to(model.device))
    
    axes[i+1].imshow(recon.cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i+1].set_title(f'Size={size}\nPerplex={perplexity.item():.1f}', fontsize=10)
    axes[i+1].axis('off')

plt.suptitle('Effect of Codebook Size on Reconstruction', fontsize=14)
plt.tight_layout()
plt.show()

print("Observations:")
print("- Larger codebooks can capture more detail")
print("- But may have lower perplexity (underutilization)")
print("- Tradeoff between expressiveness and efficiency")

## Understanding Perplexity

**Perplexity** is a crucial metric for VQ-VAE that measures codebook utilization:

$$\text{Perplexity} = \exp\left(-\sum_{k=1}^K p_k \log p_k\right)$$

Where $p_k$ is the probability that code $k$ is used.

**Interpretation:**
- **Maximum perplexity** = $K$ (all codes used equally)
- **Minimum perplexity** = 1 (only one code used)
- **Healthy VQ-VAE**: Perplexity close to $K$ indicates diverse codebook usage

Let's visualize perplexity during training.

In [ ]:
# Train a fresh model while tracking perplexity
class VQVAEWithPerplexityTracking(VQVAE):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_perplexities = []
    
    def training_step(self, batch, batch_idx):
        loss = super().training_step(batch, batch_idx)
        
        # Track perplexity
        if batch_idx % 50 == 0:
            x, _ = batch
            with torch.no_grad():
                _, _, perplexity, _ = self(x)
            self.train_perplexities.append(perplexity.item())
        
        return loss

# Train
print("Training VQ-VAE while tracking perplexity...\n")

model_tracked = VQVAEWithPerplexityTracking(
    input_dim=784,
    hidden_dim=400,
    latent_dim=20,
    num_embeddings=512,
    commitment_cost=0.25,
    learning_rate=1e-3
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto',
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)

trainer.fit(model_tracked, train_loader, test_loader)

# Plot perplexity over training
plt.figure(figsize=(10, 5))
plt.plot(model_tracked.train_perplexities, linewidth=2)
plt.axhline(y=512, color='r', linestyle='--', label='Max perplexity (512)')
plt.xlabel('Training Step (sampled every 50 batches)')
plt.ylabel('Perplexity')
plt.title('Codebook Perplexity During Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_perplexity = model_tracked.train_perplexities[-1]
print(f"\nFinal perplexity: {final_perplexity:.2f} / 512")
print(f"Codebook utilization: {final_perplexity/512*100:.1f}%")

**Healthy training**: Perplexity should stabilize at a reasonably high value. If it drops too low, you may have **codebook collapse** - most codes unused!

## Key Takeaways

### What We Learned

1. **Discrete vs Continuous Latent Spaces**
   - VAEs use continuous $z \in \mathbb{R}^d$
   - VQ-VAE uses discrete codes from a learned codebook
   - Discrete codes avoid posterior collapse and enable sharper reconstructions

2. **Vector Quantization**
   - Map continuous encoder outputs to nearest codebook vector
   - Learn codebook through training
   - Use straight-through estimator for gradients

3. **The VQ-VAE Loss**
   - Reconstruction loss: minimize $\|x - \hat{x}\|^2$
   - Codebook loss: update codebook to match encoder outputs
   - Commitment loss: keep encoder committed to chosen codes

4. **Codebook Analysis**
   - Perplexity measures codebook utilization
   - Different digits use different subsets of codes
   - Most-used codes represent common patterns

5. **Advantages of VQ-VAE**
   - No posterior collapse
   - Sharper reconstructions than VAE
   - Natural for hierarchical generation
   - Perfect for autoregressive priors (PixelCNN, Transformers)

### Why VQ-VAE Matters

VQ-VAE represents a fundamental shift in generative modeling:
- **DALL-E**: Uses VQ-VAE + Transformer for text-to-image generation
- **Jukebox**: Uses VQ-VAE for music generation
- **VQ-VAE-2**: Hierarchical extension for high-resolution images

The discrete latent space makes it possible to apply powerful sequence models (Transformers) to generation!

### Next Steps

- **Convolutional VQ-VAE**: Use CNNs for better image processing
- **VQ-VAE-2**: Hierarchical multi-scale latent codes
- **Prior learning**: Train PixelCNN or Transformer to model $p(z)$
- **Latent diffusion**: Modern alternative combining VAE + diffusion

## Congratulations!

You've mastered VQ-VAE! You now understand:
- Why discrete latent spaces are powerful
- How vector quantization works
- The straight-through estimator trick
- Codebook analysis and utilization
- The path from VQ-VAE to modern generative models

The discrete latent space is a powerful idea that's revolutionizing generative modeling - from DALL-E to modern image synthesis!